# ProFAQLM: Instruction Fine-Tuning and GGUF Quantization

End-to-end pipeline for training, merging, quantizing, and publishing custom academic instruction models tailored for citation-grounded question answering and structured exam evaluation.

- **Hugging Face Model**: [park-bit/ProFAQLM-3B-Exam](https://huggingface.co/park-bit/ProFAQLM-3B-Exam)
- **GitHub Repository**: [park-bit/ProFAQLM](https://github.com/park-bit/ProFAQLM)
- **Base Model**: `Qwen/Qwen2.5-3B-Instruct`
- **Quantization**: 4-bit NormalFloat (NF4) training, Q4_K_M GGUF inference
- **Dataset**: `allenai/sciq` (Academic Science Examination Corpus)


### 1. Install Dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q torch transformers datasets peft bitsandbytes trl accelerate huggingface_hub gguf


### 2. Environment and Workspace Initialization

In [ ]:
import os
import json
import torch
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

WORK_DIR = Path("/kaggle/working/profaqlm_workspace") if Path("/kaggle/working").exists() else Path("./profaqlm_workspace")
DATA_DIR = WORK_DIR / "data"
ADAPTER_DIR = WORK_DIR / "lora_adapter"
MERGED_DIR = WORK_DIR / "merged_model"
GGUF_DIR = WORK_DIR / "gguf_export"

for p in [DATA_DIR, ADAPTER_DIR, MERGED_DIR, GGUF_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


### 3. Ingest and Format Academic Dataset (SciQ)

In [ ]:
raw_dataset = load_dataset("allenai/sciq")

def clean_and_format(sample):
    support = sample.get("support", "").strip()
    question = sample.get("question", "").strip()
    answer = sample.get("correct_answer", "").strip()

    if not support or not question or not answer:
        return None

    instruction = "Answer the question in structured academic exam format with citations [N] based only on the provided context."
    input_text = f"Context: [1]: {support}\n\nQuestion: {question}"

    output_text = (
        f"## 1. Formal Definition\n\n"
        f"**{answer}** [1].\n\n"
        f"## 2. Core Explanation\n\n"
        f"{support} [1].\n\n"
        f"## 3. Key Takeaway\n\n"
        f"This principle demonstrates that {answer.lower()} functions directly in accordance with the documented mechanism [1]."
    )

    return {
        "instruction": instruction,
        "input": input_text,
        "output": output_text,
    }

train_records = []
for item in raw_dataset["train"]:
    rec = clean_and_format(item)
    if rec:
        train_records.append(rec)
    if len(train_records) >= 800:
        break

val_records = []
for item in raw_dataset["validation"]:
    rec = clean_and_format(item)
    if rec:
        val_records.append(rec)
    if len(val_records) >= 100:
        break

print(f"Cleaned training samples: {len(train_records)}")
print(f"Cleaned validation samples: {len(val_records)}")


### 4. Export Training and Validation Splits

In [ ]:
train_file = DATA_DIR / "train.jsonl"
val_file = DATA_DIR / "val.jsonl"

with open(train_file, "w", encoding="utf-8") as f:
    for r in train_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open(val_file, "w", encoding="utf-8") as f:
    for r in val_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Train split: {train_file}")
print(f"Validation split: {val_file}")


### 5. Load Base Model with 4-bit NormalFloat Quantization

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device_map = "auto" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    torch_dtype=torch.float16,
    device_map=device_map,
    trust_remote_code=True,
)

if torch.cuda.is_available():
    model = prepare_model_for_kbit_training(model)

print("Base model initialized in 4-bit.")


### 6. Attach LoRA Parameter-Efficient Adapter

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


### 7. Configure SFTTrainer with Chat Template

In [ ]:
def format_chat_prompt(sample):
    instruction = sample.get("instruction", "")
    input_text = sample.get("input", "")
    output_text = sample.get("output", "")

    return f"""<|im_start|>system
",
{instruction}<|im_end|>
",
<|im_start|>user
",
{input_text}<|im_end|>
",
<|im_start|>assistant
",
{output_text}<|im_end|>"""

hf_dataset = load_dataset("json", data_files={"train": str(train_file), "val": str(val_file)})
hf_dataset = hf_dataset.map(lambda x: {"text": format_chat_prompt(x)})

training_args = SFTConfig(
    output_dir=str(ADAPTER_DIR),
    dataset_text_field="text",
    max_length=2048,
    loss_type="nll",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=100,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    report_to="none",
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=hf_dataset["train"],
    eval_dataset=hf_dataset["val"],
    processing_class=tokenizer,
    args=training_args,
)

print("Trainer initialized.")


### 8. Run Supervised Fine-Tuning

In [ ]:
print("Starting training...")
trainer.train()

print(f"Saving adapter weights to: {ADAPTER_DIR}")
peft_model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("LoRA adapter saved successfully.")


### 9. Merge LoRA Weights into Standalone Model

In [ ]:
import gc
for v in ["trainer", "peft_model", "model"]:
    if v in globals():
        del globals()[v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao as torchao_module
    torchao_module.is_torchao_available = lambda: False
except Exception:
    pass

print("Loading base model in FP16 for weight merging...")
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
)
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)

print("Applying LoRA adapter...")
lora_model = PeftModel.from_pretrained(base_model_fp16, str(ADAPTER_DIR))
merged_model = lora_model.merge_and_unload()

print(f"Saving merged weights to: {MERGED_DIR}")
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
base_tokenizer.save_pretrained(str(MERGED_DIR))
print("Model merge complete.")


### 10. Convert to GGUF and Quantize to 4-bit Q4_K_M (llama.cpp)

In [ ]:
%%bash
if [ ! -d "llama.cpp" ]; then
    git clone https://github.com/ggerganov/llama.cpp
fi

cd llama.cpp
pip install -q gguf

# Convert HF weights to FP16 GGUF
python convert_hf_to_gguf.py ../profaqlm_workspace/merged_model --outfile ../profaqlm_workspace/gguf_export/profaqlm-3b-f16.gguf

# Build quantizer
cmake -B build -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=OFF
cmake --build build --config Release --target llama-quantize -j4

# Quantize to 4-bit Q4_K_M
./build/bin/llama-quantize \
  ../profaqlm_workspace/gguf_export/profaqlm-3b-f16.gguf \
  ../profaqlm_workspace/gguf_export/profaqlm-3b-q4_k_m.gguf \
  q4_k_m

ls -lh ../profaqlm_workspace/gguf_export/


### 11. Upload Artifacts to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, create_repo
import os
import getpass

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Enter Hugging Face Token (hidden): ")

api = HfApi(token=HF_TOKEN)
HF_USERNAME = api.whoami()["name"]
REPO_ID = f"{HF_USERNAME}/ProFAQLM-3B-Exam"
print(f"Target repository: {REPO_ID}")

create_repo(REPO_ID, repo_type="model", token=HF_TOKEN, exist_ok=True)

print("Uploading LoRA adapter...")
api.upload_folder(
    folder_path=str(ADAPTER_DIR),
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Add trained ProFAQLM LoRA adapter",
)

gguf_file = GGUF_DIR / "profaqlm-3b-q4_k_m.gguf"
if gguf_file.exists():
    print(f"Uploading GGUF: {gguf_file}...")
    api.upload_file(
        path_or_fileobj=str(gguf_file),
        path_in_repo="profaqlm-3b-q4_k_m.gguf",
        repo_id=REPO_ID,
        repo_type="model",
        commit_message="Add 4-bit Q4_K_M GGUF model",
    )

print(f"Upload complete: https://huggingface.co/{REPO_ID}")
